# 01 · EDA y calidad de datos — NYC 311 (API NYC Open Data)

**Proyecto:** Despliegue de soluciones analíticas (MIAD)
**Pregunta de negocio:** ¿qué nivel de demanda (bajo, medio, alto) de solicitudes no urgentes se espera la próxima semana en cada borough de Nueva York?
**Fuente:** NYC Open Data, dataset `erm2-nwe9` (311 Service Requests), consultado por API. Rango 2022-2025.
**Entrada:** `data/raw/nyc311_diario_borough_tipo.parquet` (una fila por día, borough y tipo de queja; `n` = número de solicitudes).

Ejecuta las celdas **en orden** y revisa la validación de cada una antes de seguir.

## 0. Configuración

In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})

NOMBRE = Path("data") / "raw" / "nyc311_diario_borough_tipo.parquet"

# Candidatos, en orden de prioridad
candidatos = []
if os.getenv("REPO_RAIZ"):
    candidatos.append(Path(os.environ["REPO_RAIZ"]))
candidatos += [
    Path("/work"),                                   # raíz del repo dentro del contenedor Docker
    Path("/home/leonardo/ruby/dsa-proyecto"),        # ruta en tu máquina (si Jupyter corre en el host)
    Path.cwd(),
    *Path.cwd().parents,
]

RAIZ = next((p for p in candidatos if (p / NOMBRE).exists()), None)

if RAIZ is None:
    print("No encontré el parquet en ninguna de estas rutas:")
    for p in candidatos:
        print("  -", p)
    raise FileNotFoundError(
        f"No se encontró {NOMBRE}. Revisa el volumen de Docker o haz `dvc pull`."
    )

RUTA = RAIZ / NOMBRE
FIG = RAIZ / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Cifras de referencia de la validación previa (validar_nyc_api.py)
ESPERADO_TOTAL = 13_506_495
ESPERADO_ANIO = {2022: 3_169_960, 2023: 3_224_723, 2024: 3_456_770, 2025: 3_655_042}
ESPERADO_DIAS = 1461  # 2022-01-01 a 2025-12-31

print("Raíz del repo:", RAIZ)
print("Archivo existe:", RUTA.exists())

No encontré el parquet en ninguna de estas rutas:
  - /work
  - /home/leonardo/ruby/dsa-proyecto
  - /work
  - /


FileNotFoundError: No se encontró data/raw/nyc311_diario_borough_tipo.parquet. Revisa el volumen de Docker o haz `dvc pull`.

## 1. Carga y estructura

In [2]:
df = pd.read_parquet(RUTA)
df["dia"] = pd.to_datetime(df["dia"])

print("Forma:", df.shape)
print(df.dtypes, "\n")
print(df.head())
print("\nMemoria: %.1f MB" % (df.memory_usage(deep=True).sum() / 1e6))

FileNotFoundError: /work/data/raw/nyc311_diario_borough_tipo.parquet

**Validación:** 4 columnas (`borough`, `dia`, `complaint_type`, `n`) y cientos de miles de filas (~667.000 estimadas).

## 2. Calidad: nulos, duplicados y valores inválidos

In [ ]:
clave = ["borough", "dia", "complaint_type"]

calidad = pd.Series({
    "filas": len(df),
    "nulos (total celdas)": int(df.isna().sum().sum()),
    "duplicados por clave": int(df.duplicated(clave).sum()),
    "n <= 0": int((df["n"] <= 0).sum()),
    "n no entero": int((df["n"] % 1 != 0).sum()),
    "borough vacío": int((df["borough"].str.strip() == "").sum()),
    "complaint_type vacío": int((df["complaint_type"].str.strip() == "").sum()),
})
print(calidad.to_string())

total = int(df["n"].sum())
print(f"\nSolicitudes totales: {total:,}  |  esperado: {ESPERADO_TOTAL:,}  |  coincide: {total == ESPERADO_TOTAL}")

por_anio = df.groupby(df["dia"].dt.year)["n"].sum()
comp = pd.DataFrame({"obtenido": por_anio, "esperado": pd.Series(ESPERADO_ANIO)})
comp["coincide"] = comp["obtenido"] == comp["esperado"]
print("\n", comp)

**Validación:** cero nulos, duplicados y `n <= 0`. El total y cada año deben coincidir con la validación previa (`coincide = True`). Si no coinciden, la descarga está incompleta: repítela antes de seguir.

## 3. Boroughs

In [ ]:
b = df.groupby("borough")["n"].sum().sort_values(ascending=False)
tabla_b = pd.DataFrame({"solicitudes": b, "pct": (b / b.sum() * 100).round(2)})
print(tabla_b)

fig, ax = plt.subplots(figsize=(7, 3.5))
tabla_b["solicitudes"].plot.barh(ax=ax, color="#4c72b0")
ax.invert_yaxis(); ax.set_title("Solicitudes por borough (2022-2025)"); ax.set_xlabel("solicitudes")
fig.savefig(FIG / "boroughs.png", bbox_inches="tight"); plt.show()

**Validación:** cinco boroughs más `Unspecified` (~0,1%). Brooklyn ~30% y Staten Island ~4%: escalas muy distintas, por eso los niveles bajo/medio/alto deben calcularse **por borough**. `Unspecified` se excluye de la serie semanal.

## 4. Cobertura temporal

In [ ]:
rango = pd.date_range("2022-01-01", "2025-12-31")
dias_datos = df["dia"].drop_duplicates()
faltantes = rango.difference(dias_datos)

print("Rango en datos:", df["dia"].min().date(), "->", df["dia"].max().date())
print(f"Días con datos: {dias_datos.nunique()} (esperados: {ESPERADO_DIAS})")
print("Días faltantes:", len(faltantes), list(faltantes[:10].date))

# Combinaciones día-borough ausentes (solo boroughs principales)
principales = [x for x in df["borough"].unique() if x != "Unspecified"]
for bo in principales:
    nd = df.loc[df["borough"] == bo, "dia"].nunique()
    print(f"  {bo:<14} {nd} días ({ESPERADO_DIAS - nd} sin registros)")

mensual = df.groupby(df["dia"].dt.to_period("M"))["n"].sum()
print("\nMeses con datos:", len(mensual), "(esperados: 48)")

**Validación:** 1.461 días y 48 meses. Si faltan días, anótalos en el reporte (una semana con días vacíos no se puede comparar con las demás).

## 5. Tipos de queja y consistencia de nombres

In [ ]:
tipos = df.groupby("complaint_type")["n"].sum().sort_values(ascending=False)
norm = df["complaint_type"].str.strip().str.lower()

print("Tipos distintos:", len(tipos))
print("Tipos tras normalizar (minúsculas y espacios):", norm.nunique())
print("Top 10 = %.1f%% de las solicitudes" % (tipos.head(10).sum() / tipos.sum() * 100))
print("Tipos con menos de 1.000 solicitudes:", int((tipos < 1000).sum()))

# Familias por prefijo (p. ej. 'Noise - Residential' -> 'noise')
familia = (norm.str.split(" - ").str[0].str.strip())
fam = df.assign(familia=familia).groupby("familia")["n"].sum().sort_values(ascending=False)
print("\nFamilias de queja:", len(fam))
print(fam.head(12))

print("\nFamilia Noise:")
print(tipos[tipos.index.str.contains("noise", case=False)])

fig, ax = plt.subplots(figsize=(8, 5))
tipos.head(15).iloc[::-1].plot.barh(ax=ax, color="#55a868")
ax.set_title("Top 15 tipos de queja (2022-2025)"); ax.set_xlabel("solicitudes")
fig.savefig(FIG / "top_tipos_queja.png", bbox_inches="tight"); plt.show()

**Validación:** si el número de tipos baja al normalizar, había duplicados por mayúsculas o espacios (la limpieza los unifica). `Noise` aparece dividido en varios subtipos: conviene tratarlos como una familia.

## 6. Serie semanal por borough

In [ ]:
d = df[df["borough"] != "Unspecified"].copy()
d["semana"] = d["dia"].dt.to_period("W-SUN").dt.start_time

# Semana completa = lunes a domingo dentro del rango de fechas (criterio por calendario)
ini, fin = df["dia"].min(), df["dia"].max()
semanas = d["semana"].drop_duplicates().sort_values()
completas = semanas[(semanas >= ini) & (semanas + pd.Timedelta(days=6) <= fin)]
print(f"Semanas completas: {len(completas)} (esperadas: 208) | parciales descartadas: {len(semanas) - len(completas)}")

sem = (d[d["semana"].isin(completas)]
       .groupby(["semana", "borough"], as_index=False)["n"].sum())

print("Filas semana-borough:", len(sem), "(esperadas: 1.040)")
print("Semanas por borough:\n", sem.groupby("borough")["semana"].nunique())

fig, ax = plt.subplots(figsize=(11, 4.2))
for bo, g in sem.groupby("borough"):
    ax.plot(g["semana"], g["n"], label=bo, lw=1.2)
ax.set_title("Solicitudes semanales por borough (2022-2025)")
ax.set_ylabel("solicitudes por semana"); ax.legend(ncol=5, fontsize=8, loc="upper left")
fig.savefig(FIG / "serie_semanal_borough.png", bbox_inches="tight"); plt.show()

**Validación:** 208 semanas por borough y 1.040 filas en total. Esta es la tabla que alimentará el modelo.

## 7. Tendencia anual

In [ ]:
sem["anio"] = sem["semana"].dt.year
prom = sem.groupby(["anio", "borough"])["n"].mean().unstack().round(0)
print("Promedio semanal por año y borough:\n", prom)

tot_anio = sem.groupby("anio")["n"].sum()
print("\nCrecimiento total vs 2022:")
print(((tot_anio / tot_anio.iloc[0] - 1) * 100).round(1).astype(str) + " %")

crec = ((prom.loc[2025] / prom.loc[2022] - 1) * 100).round(1)
print("\nCrecimiento 2025 vs 2022 por borough (%):\n", crec)

fig, ax = plt.subplots(figsize=(7, 3.8))
prom.plot(ax=ax, marker="o")
ax.set_title("Promedio semanal por año"); ax.set_ylabel("solicitudes por semana")
fig.savefig(FIG / "tendencia_anual.png", bbox_inches="tight"); plt.show()

**Validación:** la demanda crece de un año a otro (~15% entre 2022 y 2025 en total). Consecuencia de diseño: terciles **fijos** calculados con 2022-2023 harían que casi todo 2025 sea "alto". Considera definir el nivel respecto a un promedio móvil de 52 semanas del mismo borough.

## 8. Estacionalidad

In [ ]:
tot = sem.groupby("semana", as_index=False)["n"].sum()
tot["anio"] = tot["semana"].dt.year
tot["sem_anio"] = tot["semana"].dt.isocalendar().week.astype(int).clip(upper=52)

fig, ax = plt.subplots(figsize=(10, 4))
for a, g in tot.groupby("anio"):
    ax.plot(g["sem_anio"], g["n"], label=str(a), lw=1.2)
ax.set_title("Solicitudes semanales por semana del año (total 5 boroughs)")
ax.set_xlabel("semana del año"); ax.set_ylabel("solicitudes"); ax.legend()
fig.savefig(FIG / "estacionalidad_semana_anio.png", bbox_inches="tight"); plt.show()

# Índice mensual: mes / promedio del año (quita la tendencia)
m = df.groupby([df["dia"].dt.year.rename("anio"), df["dia"].dt.month.rename("mes")])["n"].sum().unstack(0)
indice = (m / m.mean()).round(3)
print("Índice mensual (1,0 = promedio del año):\n", indice)

fig, ax = plt.subplots(figsize=(8, 3.8))
indice.mean(axis=1).plot.bar(ax=ax, color="#c44e52")
ax.axhline(1, color="k", lw=.8); ax.set_title("Índice estacional mensual promedio"); ax.set_xlabel("mes")
fig.savefig(FIG / "indice_estacional.png", bbox_inches="tight"); plt.show()

**Validación:** ¿se repite el patrón entre años? Si sí, `semana del año` y `mes` son buenas variables para el modelo.

## 9. Valores atípicos

In [ ]:
def z_robusto(x):
    med = x.rolling(13, center=True, min_periods=5).median()
    res = x - med
    mad = (res - res.median()).abs().median() * 1.4826
    return res / mad

sem = sem.sort_values(["borough", "semana"])
sem["z"] = sem.groupby("borough")["n"].transform(z_robusto)

atip = sem[sem["z"].abs() > 3].sort_values("z", key=abs, ascending=False)
print("Semanas con |z robusto| > 3:", len(atip))
print(atip.head(15)[["semana", "borough", "n", "z"]].round({"z": 2}).to_string(index=False))

top_meses = df.groupby(df["dia"].dt.to_period("M"))["n"].sum().sort_values(ascending=False).head(3)
print("\nMeses más altos:\n", top_meses)

**Validación:** no elimines atípicos sin entender la causa. Anótalos como hallazgo.

### 9.1 Enero de 2025 (mes más alto de la serie)

In [ ]:
def mes(anio, m):
    x = df[(df["dia"].dt.year == anio) & (df["dia"].dt.month == m)]
    return x

e24, e25 = mes(2024, 1), mes(2025, 1)
print(f"Enero 2024: {e24['n'].sum():,} | Enero 2025: {e25['n'].sum():,} | variación: {(e25['n'].sum()/e24['n'].sum()-1)*100:.1f}%")

dif = (e25.groupby("complaint_type")["n"].sum()
       .sub(e24.groupby("complaint_type")["n"].sum(), fill_value=0)
       .sort_values(ascending=False))
print("\nTipos que más aumentaron (ene 2025 - ene 2024):")
print(dif.head(8).astype(int))

print("\nPor borough:")
print((e25.groupby("borough")["n"].sum() - e24.groupby("borough")["n"].sum()).astype(int))

diario = e25.groupby("dia")["n"].sum()
print("\nDías más altos de enero 2025:")
print(diario.sort_values(ascending=False).head(5).astype(int))

**Validación:** identifica si el aumento se concentra en uno o dos tipos (p. ej. calefacción o ruido) o en pocos días. Eso decide si se trata como evento puntual o como patrón.

## 10. Candidatos a solicitudes "no urgentes"

In [ ]:
cand = tipos.head(40).reset_index()
cand.columns = ["complaint_type", "solicitudes"]
cand["pct"] = (cand["solicitudes"] / tipos.sum() * 100).round(2)
cand["pct_acum"] = cand["pct"].cumsum().round(1)
cand["no_urgente"] = ""   # <- el equipo completa: SI / NO / DUDA
print(cand.to_string(index=False))

cand.to_csv(RAIZ / "reports" / "candidatos_no_urgentes.csv", index=False)
print("\nGuardado: reports/candidatos_no_urgentes.csv")

**Decisión del equipo (obligatoria):** 311 no trae un campo de urgencia. Completa la columna `no_urgente` del CSV (SI / NO / DUDA), con el criterio y quién lo aprobó. Propuesta inicial para discutir:

- **Candidatos a no urgentes:** `Illegal Parking`, `Blocked Driveway`, `Abandoned Vehicle`, `Derelict Vehicles`, `Dirty Condition`, `Missed Collection` y la familia `Noise`.
- **Dudosos (afectan habitabilidad):** `HEAT/HOT WATER`, `WATER LEAK`, `PLUMBING`, `UNSANITARY CONDITION`.
- **Sensibles:** `Encampment`, `Homeless Person Assistance`.

## 11. Resumen de hallazgos y decisiones

In [ ]:
tot_total = int(df["n"].sum())
unspec = df.loc[df["borough"] == "Unspecified", "n"].sum()
crec_total = (tot_anio.iloc[-1] / tot_anio.iloc[0] - 1) * 100

resumen = pd.DataFrame([
    ("Nulos / duplicados / n<=0", f"{int(df.isna().sum().sum())} / {int(df.duplicated(clave).sum())} / {int((df['n']<=0).sum())}", "Sin acción si son 0"),
    ("Total de solicitudes", f"{tot_total:,}", "Coincide con la validación" if tot_total == ESPERADO_TOTAL else "REVISAR: no coincide"),
    ("Días con datos", f"{dias_datos.nunique()} de {ESPERADO_DIAS}", "Documentar faltantes"),
    ("Borough Unspecified", f"{unspec:,} ({unspec/tot_total*100:.2f}%)", "Excluir de la serie semanal"),
    ("Tipos de queja", f"{len(tipos)} (tras normalizar: {norm.nunique()})", "Normalizar y agrupar por familia"),
    ("Semanas completas", f"{len(completas)}", "Descartar semanas parciales"),
    ("Crecimiento 2022 → 2025", f"{crec_total:.1f}%", "Niveles relativos a promedio móvil 52 semanas"),
    ("Semanas atípicas (|z|>3)", f"{len(atip)}", "Revisar y documentar; no eliminar sin causa"),
    ("Regla de no urgentes", "Pendiente", "Equipo define y justifica"),
], columns=["hallazgo", "valor", "decisión"])
print(resumen.to_string(index=False))
resumen.to_csv(RAIZ / "reports" / "resumen_calidad.csv", index=False)

**Siguiente paso:** pasar estas decisiones a `src/limpieza.py` (stage de DVC `limpieza`) y generar `data/processed/nyc311_semanal_borough.parquet`.